In [1]:
# using python 3.9 environment

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split

import torch
from torchvision import datasets
from torchvision import transforms
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader

import os
import cv2

import glob 
import cuvis


In [24]:

def loadMeasurement(userSettingsDir, measurementLoc):

    print("loading user settings...")
    settings = cuvis.General(userSettingsDir)
    settings.set_log_level("info")

    print("loading measurement file...")
    mesu = cuvis.Measurement(measurementLoc)
    print("Data 1 {} t={}ms mode={}".format(mesu.name,mesu.integration_time,mesu.processing_mode.name,))

    # if not isinstance(mesu.MeasurementFlags, list):
    #     mesu.MeasurementFlags = [mesu.MeasurementFlags]

    if isinstance(mesu.measurement_flags, cuvis.MeasurementFlags):
        print(f"Flags: {mesu.measurement_flags}")
        for v in cuvis.MeasurementFlags.supremum():
            print(f'{v}: {v in mesu.measurement_flags}')
    
    cube = mesu.data.get("cube", None)
    if cube is None:
        raise Exception("Cube not found")
    cube = cube.array
    print(cube.shape)
    # cube_reshaped = cube.reshape(cube.shape[2], cube.shape[0], cube.shape[1])
    cube_result = np.transpose(cube, (2, 0, 1))
    print(cube_result.shape)
    cube = cube_result
    if cube is None:
        raise Exception("Cube not found")
    
    print("Cube shape: ", cube.shape)
    return cube

   



class AutoencoderDataset(Dataset):
    def __init__(self,  dataset_name, image_set,  root_dir, transform=None):
        self.dataset_name = dataset_name
        self.image_set = image_set
        self.transform = transform
        self.root = root_dir
        self.img_dir = root_dir
        self.img_names = [f for f in os.listdir(self.img_dir) if f.endswith('.' + 'cu3')]

        self.num_images = len( self.img_names  )

    def __getitem__(self, index):
        
        hyperspectral_image1 =self.root + self.img_names[index]
        print(hyperspectral_image1)
        userSettingsDir = "/workspaces/HyperTools/settings/ultris20/" 
        hyperImage = loadMeasurement(userSettingsDir,hyperspectral_image1)

        return hyperImage

    def __len__(self):
        return self.num_images
    
        
# Creating a PyTorch class
# 28*28 ==> 9 ==> 28*28
class spectral_AE(torch.nn.Module):
    def __init__(self):
        super().__init__()

        # Building an linear encoder with Linear
        # layer followed by Relu activation function
        # 140 ==> 3
        self.encoder = torch.nn.Sequential(
            torch.nn.Linear(hyper_2d.shape[1], 64),
            torch.nn.ReLU(),
            torch.nn.Linear(64, 32),
            torch.nn.ReLU(),
            torch.nn.Linear(32, 16),
            torch.nn.ReLU(),
            torch.nn.Linear(16, 8),
            torch.nn.ReLU(),
            torch.nn.Linear(8, 3),
        )

        # Building an linear decoder with Linear
        # layer followed by Relu activation function
        # The Sigmoid activation function
        # outputs the value between 0 and 1
        # 3 ==> 140
        self.decoder = torch.nn.Sequential(
            torch.nn.Linear(3, 8),
            torch.nn.ReLU(),
            torch.nn.Linear(8, 16),
            torch.nn.ReLU(),
            torch.nn.Linear(16, 32),
            torch.nn.ReLU(),
            torch.nn.Linear(32, 64),
            torch.nn.ReLU(),
            torch.nn.Linear(64, hyper_2d.shape[1]),
            torch.nn.Sigmoid()
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded    



In [29]:
input_data_dir = "/workspaces/HyperImages/cornfields/session_002/" # need to reprocess this data

input_data_dir = "/workspaces/HyperImages/cornflakes_and_pasta/cornflakes_and_noodles_000/"



train_dataset = AutoencoderDataset(dataset_name='cornfields', root_dir=input_data_dir, image_set='train')

example= train_dataset[3]

/workspaces/HyperImages/cornflakes_and_pasta/cornflakes_and_noodles_000/cornflakes_and_noodles_000_002_snapshot.cu3
loading user settings...
loading measurement file...
Data 1 cornflakes_and_noodles_000_002_snapshot t=125.0ms mode=Reflectance
Flags: MeasurementFlags([])
POOR_REFERENCE: False
OVERILLUMINATED: False
DARK_TEMP: False
WHITE_INTTIME: False
WHITE_TEMP: False
WHITEDARK_INTTIME: False
WHITEDARK_TEMP: False
(275, 290, 51)
(51, 275, 290)
hi (51, 275, 290)
Cube shape:  (51, 275, 290)


In [ ]:
spectral_dataset = SpectralDataset(hyper_2d)
spectral_loader = DataLoader(spectral_dataset, batch_size=2048, shuffle=True)

spectral_train_data, spectral_test_data = train_test_split(
    spectral_dataset,  test_size=0.3, random_state=21
)

min_val = torch.min(torch.from_numpy(hyper_2d))
max_val = torch.max(torch.from_numpy(hyper_2d))

# spectral_train_data = ((torch.from_numpy(spectral_train_data) - min_val) / (max_val - min_val)).float()
# spectral_test_data = ((torch.from_numpy(spectral_test_data) - min_val) / (max_val - min_val)).float()

# Normalize each Tensor in the list
spectral_train_data = [(data - min_val) / (max_val - min_val) for data in spectral_train_data]
spectral_test_data = [(data - min_val) / (max_val - min_val) for data in spectral_test_data]

# Convert the list of Tensors to a single Tensor
spectral_train_data = torch.stack(spectral_train_data).float()
spectral_test_data = torch.stack(spectral_test_data).float()




In [ ]:
spectral_model = spectral_AE()
train_loader = DataLoader(spectral_train_data, batch_size=16384, shuffle=True)
test_loader = DataLoader(spectral_test_data, batch_size=16384)


In [ ]:
loss_function = torch.nn.L1Loss()
criterion = torch.nn.MSELoss()
device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# print (device)
spectral_model = spectral_model.to(device)
optimizer = torch.optim.Adam(spectral_model.parameters(),
                             lr = 1e-4,
                             weight_decay = 1e-5)

epochs = 20
history_loss_train = []
history_loss_val = []
for epoch in range(epochs):
    spectral_model.train()
    for inputs in train_loader:
        inputs = inputs.to(device)
        outputs = spectral_model(inputs)
        loss = criterion(outputs, inputs)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    spectral_model.eval()
    with torch.no_grad():
        for inputs in test_loader:
            inputs = inputs.to(device)
            outputs = spectral_model(inputs)
            val_loss = criterion(outputs, inputs)
    history_loss_train.append(loss.item())
    history_loss_val.append(val_loss.item())
    print(f'Epoch {epoch+1}/{epochs}, Loss: {loss.item()}, Validation Loss: {val_loss.item()}')